# pmlab tutorial: from a market list to a paper-traded champion

One notebook, the whole workflow, **offline**: *discover → panel → tune → backtest → gate → paper → report*.
It uses only pmlab's public API (everything is imported from `pmlab`) on a small sample dataset that ships inside the package, so it runs the same on your laptop, in CI and in a fresh `pip install pmlab`. It runs in roughly 15 seconds.

> **The data is synthetic.** `pmlab.datasets.load_sample()` returns 750 weather-temperature markets shaped like Polymarket's "Highest temperature in *city* on *date*?", generated by a seeded script. It is for learning how the pieces fit, not a statement about real markets: the sample book is deliberately beatable, so the paper PnL below says nothing about what you would earn.

Three rules from `AGENTS.md` are enforced by the library, and this notebook runs into each of them on purpose:

1. **No NO_GO champion is ever published.** `ChampionManifest.publish` raises on a failed gate. It is not configurable.
2. **No lookahead.** A price is only used if it was observed before its decision, a model only trains on the past, and a trade settles only once the truth is final.
3. **Workspaces never mix.** `ops_daily` is live forward evidence and `historical_real` is training data; every path comes from a `WorkspaceContext`.

Install with `pip install "pmlab[tune]"` (the `tune` extra brings Optuna, used in step 3).

In [ ]:
import tempfile
import warnings
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

import pmlab
from pmlab import (
    BacktestConfig,
    ChampionManifest,
    CostModel,
    EdgeSignal,
    HoldoutGateResult,
    LGBMForecaster,
    PaperBroker,
    SettlementEngine,
    TradePolicy,
    TunedForecaster,
    WorkspaceContext,
    build_panel,
    capture_backtest,
    check_no_leakage,
    generate_report,
    reproduce,
    resolve_seeds,
    run_backtest,
)
from pmlab.datasets import load_sample

warnings.filterwarnings("ignore", message="IProgress not found")  # Optuna's progress bar, cosmetic
print("pmlab", pmlab.__version__)

## 0. A workspace for everything we write

All artifacts (champion, trade log, report) go under a workspace, never next to the code. Here the workspace lives in a temporary directory, so the notebook leaves nothing behind. On a real machine you would run commands as `scripts/pmlab-workspace <name> <cmd>`, which sets the environment so `WorkspaceContext.from_env()` resolves the same paths.

In [ ]:
root = Path(tempfile.mkdtemp(prefix="pmlab-tutorial-"))
ws = WorkspaceContext.from_name("tutorial", repo_root=root)
live_ws = WorkspaceContext.from_name("ops_daily", repo_root=root)  # live evidence, never ours to write
ws.artifacts_root.mkdir(parents=True)

# two workspaces never share a path (neither contains the other), so nothing can leak between them
assert not live_ws.artifacts_root.is_relative_to(ws.artifacts_root)
assert not ws.artifacts_root.is_relative_to(live_ws.artifacts_root)
print("artifacts:", ws.artifacts_root.relative_to(root))

## 1. Discover

`load_sample()` verifies the bundled file against its checksum and returns the dataset. `sample.plugin(day)` is the **real** `WeatherTmaxPlugin`: its constructor takes its Gamma, forecast and truth clients by injection, and here those clients read the bundled file instead of the network. Nothing about the plugin is subclassed or switched off, so `discover_markets()` runs its production code.

The replay is by morning. A plugin built for a `day` stamps its prices with 07:30 UTC that day and can only see observations that were final by then.

In [ ]:
sample = load_sample()
print(sample)

day = sample.days[0]
plugin = sample.plugin(day)
today = plugin.discover_markets()  # offline: the markets open for tomorrow
spec = today[0]
print(spec.question, "| closes", spec.close_time)
print("prices seen at", spec.metadata["price_observed_at"])
pd.DataFrame(spec.metadata["outcome_schema"]).T

Truth is not available the morning a market is discovered (tomorrow has not happened), and `is_truth_final` says so. That is rule 2 at work: nothing may settle before it is final.

In [ ]:
print("truth that morning:", plugin.fetch_truth(spec), "| final:", plugin.is_truth_final(spec))

resolver = sample.plugin()  # the same plugin once the whole sample has resolved: it labels and settles
print("truth afterwards:  ", resolver.fetch_truth(spec), "| final:", resolver.is_truth_final(spec))

A daily job would discover once each morning. Replaying that over the sample gives every market with its own observation time:

In [ ]:
discovered = [(day, spec) for day in sample.days for spec in sample.plugin(day).discover_markets()]
specs = [spec for _, spec in discovered]
print(len(specs), "markets,", len(sample.cities), "cities,", sample.days[0], "to", sample.days[-1])

## 2. Panel

`build_training_rows` turns a market into **one row per temperature bin**: that bin's own price, the forecast features, and whether it won. We decide at 12:00 UTC the day before the target day, hours after the 07:30 prices were seen. A price seen at or after the decision is excluded and counted, never used:

In [ ]:
exclusions = {}
too_early = resolver.build_training_rows(
    spec, "previous_evening", decision_time=f"{day}T07:30:00+00:00", exclusions=exclusions
)
print(too_early, exclusions)  # no rows: the price was seen AT the decision, not before it

In [ ]:
exclusions = {}
rows = [
    resolver.build_training_rows(
        spec, "previous_evening", decision_time=f"{day}T12:00:00+00:00", exclusions=exclusions
    )
    for day, spec in discovered
]
panel = build_panel(rows, check_leakage=True)  # LeakageError: a late price, or a feature that gives the outcome away
print("excluded:", exclusions or "nothing", "| panel:", panel.shape)
panel.head(9)[["market_id", "decision_date", "outcome_label", "winning_label", "market_price"]]

The plugin's forecast features are the same for every bin of a market, so by themselves they cannot tell bins apart. We add features that are relative to the bin (how far it sits from the forecast and from climatology, whether it is a tail) with plain pandas, plus the bin's own market price, which was known at decision time.

Then the leakage check. `check_no_leakage` scores each `feature_*` column against what the model is trained on and flags any that separate it too well to be honest. On a one-row-per-bin panel that is "did this bin win" (`outcome_label == winning_label`), so the check needs no label from us; `build_panel(rows, check_leakage=True)` above ran the same check on the plugin's own features.

In [ ]:
meta = {(s.market_id, b.label): (s.metadata["city"], b) for s in specs for b in s.outcome_bins}


def bin_centre(bin_):
    if bin_.lower is not None and bin_.upper is not None:
        return (bin_.lower + bin_.upper) / 2
    return bin_.upper - 1.0 if bin_.upper is not None else bin_.lower + 1.0


pairs = list(zip(panel["market_id"], panel["outcome_label"], strict=True))
centre = pd.Series([bin_centre(meta[k][1]) for k in pairs], index=panel.index)

panel["segment"] = [meta[k][0] for k in pairs]  # the city: what the gate groups trades by
panel["feature_bin_dist"] = centre - panel["feature_fcst_tmax"]
panel["feature_bin_z"] = panel["feature_bin_dist"] / panel["feature_ens_spread"]
panel["feature_bin_dist_climo"] = centre - panel["feature_climo_tmax"]
panel["feature_is_tail"] = panel["outcome_label"].str.contains("<=|>=").astype(float)
panel["feature_market_price"] = panel["market_price"]
features = [c for c in panel.columns if c.startswith("feature_")]

leak = check_no_leakage(panel, raise_on_leak=False)
print(len(features), "features | leaked:", leak.leaked or "none")
print("strongest single-feature separation:", round(max(leak.scores.values()), 3))

## 3. Tune

The 150 days split into three windows that never overlap, which is what keeps the numbers honest:

| window | days | used for |
|---|---|---|
| tune | 1-60 | Optuna searches the hyperparameters here, with a walk-forward score per trial |
| backtest | 61-120 | walk-forward evaluation of the tuned model, then the gate |
| live | 121-150 | paper trading by a champion that never saw these days |

`TunedForecaster` scores every trial with `rolling_origin_eval` (train on the past, predict the next date), so a trial cannot peek ahead either.

It also has to score trials by the rule that will be traded. By default a trial is the historical backtest (YES on the most probable bin of every market, no threshold), which is not what the paper broker does: it takes, per market, the best after-fee edge, YES or NO, if it clears `min_edge`. So the tuner is handed the **live policy** and the **costs** the broker will pay (`policy=`, `costs=`), the same two objects step 4 backtests with. Tuned by the default rule, a model can win the search and still be the wrong one under this policy ([docs/tuning.md](https://github.com/ArtBreguez/polymarket-lab/blob/main/docs/tuning.md)).

In [ ]:
dates = sorted(panel["decision_date"].unique())
tune_end, backtest_end = dates[60], dates[120]
tune_panel = panel[panel["decision_date"] < tune_end]
backtest_panel = panel[panel["decision_date"] < backtest_end].reset_index(drop=True)
live_days = dates[120:]
print(f"tune {dates[0]}..{dates[59]} | backtest ..{dates[119]} | live {live_days[0]}..{live_days[-1]}")


def build(params):
    return LGBMForecaster(random_state=7, n_jobs=1, **params)


def space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 60, 200, step=20),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 4, 24),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
    }


policy = TradePolicy.live(min_edge=0.05)  # the rule the paper broker will trade with...
costs = CostModel(taker_bps=30.0, slippage_bps=50.0)  # ...and the costs it will pay

tuner = TunedForecaster(
    build,
    space,
    n_trials=8,
    metric="total_pnl",
    random_state=7,
    stride=10,
    min_train_rows=500,
    policy=policy,
    costs=costs,
).tune(tune_panel)
best = tuner.best_params_
print("best:", best, "| tuning score:", round(tuner.best_score_, 2))

## 4. Backtest, with a reproducibility record

The backtest walks forward over the backtest window: for each decision date it trains on every earlier row, predicts that date, and trades what the **live policy** would trade (per market the best after-fee edge, YES or NO, if it clears `min_edge`), priced with the same `CostModel` the paper broker will use. `min_train_rows=2700` is the 60 tuning days (60 days × 5 cities × 9 bins), so no date inside the tuning window is ever graded with parameters tuned on it.

`run_backtest` seeds every generator, and `capture_backtest` records the seeds, library versions, resolved config and a fingerprint of the data, so the result can be re-run later (`pmlab reproduce`). The record also carries the objective the parameters were tuned under (`tuning=tuner.tuning_config()`: metric, policy, costs) and the sampler seed (`optuna`), so the search can be repeated too.

In [ ]:
config = BacktestConfig.resolve(
    plugin="weather_tmax",
    model_params=best,
    stride=1,
    min_train_rows=2700,
    costs=costs,
    policy=policy,
    tuning=tuner.tuning_config(),
)
seeds = resolve_seeds(7, optuna=tuner.random_state)
outcome = run_backtest(backtest_panel, config, seeds)
record = capture_backtest(backtest_panel, config, seeds, outcome.metrics, source="tutorial sample")

trades = outcome.trades
print({k: round(v, 3) if isinstance(v, float) else v for k, v in outcome.metrics.items()})
trades.groupby("segment")["realized_pnl"].agg(trades="count", pnl="sum", per_trade="mean").round(2)

## 5. Gate

`HoldoutGateResult.evaluate` grades each city on its own: enough trades, and a non-negative total PnL. Every city we ask about has to pass, otherwise the decision is **NO_GO**.

In [ ]:
gate_all = HoldoutGateResult.evaluate(
    trades, required_segments=sample.cities, min_trades_per_segment=30
)
print("decision:", gate_all.decision)
pd.DataFrame([vars(r) for r in gate_all.segment_results]).set_index("segment")

One city fails, so the gate says NO_GO, and publishing is refused outright. Rule 1: this is a `ValueError`, not a warning, and there is no flag to skip it.

The model we would have published is fitted on the backtest window only, so it knows nothing about the live days.

In [ ]:
bt_won = backtest_panel["outcome_label"] == backtest_panel["winning_label"]
champion = build(best)
champion.fit(backtest_panel[features], bt_won.astype(int))

champion_dir = ws.artifacts_root / "champion"
try:
    ChampionManifest.publish(champion, gate_all, champion_dir, plugin_family="weather_tmax")
except ValueError as err:
    print("refused:", err)
assert not (champion_dir / "champion.json").exists()  # nothing was written

A failed gate is something to investigate, not bypass. Here the table says which city lost money: in the sample, Lisbon's book prices the day more sharply than our forecast does, so trading against it loses (the generator makes that so on purpose; with real markets you would go and find out why).

What we may do, in the open, is **scope the champion to the cities that cleared the gate** and let the paper broker enforce it. The segments that passed are written into `champion.json`, which stays the only source the broker reads. Scoping after looking at the gate is a choice made on the backtest window, so it still has to survive data the choice never saw, and the live window is exactly that.

In [ ]:
passing = [r.segment for r in gate_all.segment_results if r.passes]
gate = HoldoutGateResult.evaluate(trades, required_segments=passing, min_trades_per_segment=30)
print(gate.decision, "on", passing)

manifest = ChampionManifest.publish(
    champion, gate, champion_dir, plugin_family="weather_tmax", costs=costs, reproducibility=record
)
print("published; allowed segments:", sorted(manifest.get_allowed_segments()))

The champion carries the record of the backtest that justified it. Re-executing that backtest from the record alone must give the same numbers (this is what `pmlab reproduce champion.json` does from the command line):

In [ ]:
loaded = ChampionManifest.load(champion_dir / "champion.json")
report = reproduce(loaded.reproducibility, panel=backtest_panel)
print(report.status, "|", len(report.comparisons), "metrics compared | data matches the record:", report.data_match)
assert report.reproduced

## 6. Paper trading

The live window plays out day by day with the champion loaded **from `champion.json` alone**, as a daily job would. For every market it scores each bin at its decision-time price, turns the best one into an `EdgeSignal` (edges computed exactly as the backtest computes them), and hands the signals to a `PaperBroker` restricted to the champion's allowed segments.

The broker refuses a signal whose horizon cutoff has passed, one it has already recorded, and one outside the allowed segments.

In [ ]:
model = loaded.load_model()
live = panel[panel["decision_date"].isin(live_days)].copy()
live["prob"] = model.predict_proba(live[features])[:, 1]

broker = PaperBroker(
    trades_path=ws.artifacts_root / "paper_trades.json",
    allowed_segments=loaded.get_allowed_segments(),
    costs=costs,
)
by_market = {spec.market_id: spec for _, spec in discovered}
considered, recorded = Counter(), []
for day in live_days:
    signals = []
    for market_id, rows in live[live["decision_date"] == day].groupby("market_id"):
        spec = by_market[market_id]
        candidates = [
            EdgeSignal.from_quote(
                market_id=market_id,
                city_or_segment=spec.metadata["city"],
                target_date=spec.metadata["target_date"],
                horizon="previous_evening",
                outcome_label=r.outcome_label,
                yes_price=r.market_price,
                model_prob=r.prob,
                costs=costs,
                stake=1.0,
            )
            for r in rows.itertuples()
        ]
        best_signal = max((c for c in candidates if c), key=lambda c: c.best_edge)
        if best_signal.best_edge >= 0.05:
            signals.append(best_signal)
            considered[best_signal.city_or_segment] += 1
    noon = datetime.fromisoformat(f"{day}T12:00:00+00:00")
    recorded += broker.record(signals, now_utc=noon, city_timezones=sample.city_timezones)

print("signals worth trading:", dict(considered))
print("recorded:", dict(Counter(t["city_or_segment"] for t in recorded)))
assert "Lisbon" in considered and not any(t["city_or_segment"] == "Lisbon" for t in recorded)

Lisbon produced signals but none were recorded: the broker applied the champion's segment list. Now settlement. A trade settles only once `plugin.is_truth_final(...)` is true. We first settle from a morning in the middle of the live window, claiming every trade is due, and only the trades whose day had really ended are settled. The rest stay open. Then we settle again once the sample is complete.

In [ ]:
live_specs = [by_market[m] for m in live["market_id"].unique()]
due = sample.last_target_date  # "every trade is due": only is_truth_final decides
now = datetime(2025, 9, 1, tzinfo=UTC)

halfway = live_days[len(live_days) // 2]
early = SettlementEngine(sample.plugin(halfway), ws.artifacts_root / "paper_trades.json")
print("on the morning of", halfway, early.settle_all(live_specs, today_str=due, now_utc=now))

final = SettlementEngine(sample.plugin(), ws.artifacts_root / "paper_trades.json")
print("once the sample is complete:", final.settle_all(live_specs, today_str=due, now_utc=now))

## 7. Report

The trade log is plain JSON, and `generate_report` renders it as one self-contained HTML page (equity curve, per-city table, every trade). Before looking at it, the two numbers that matter: did the champion do what the backtest promised, and were its probabilities any good?

In [ ]:
paper = broker.load_trades()
settled = pd.DataFrame([t for t in paper if t["realized_pnl"] is not None])
assert len(settled) == len(paper) > 0, "every paper trade should have settled"

occurred = (settled["outcome"] == "won").astype(float)
brier = float(((settled["model_prob"] - occurred) ** 2).mean())
bt = trades[trades["segment"].isin(passing)]

for name, t in (("backtest", bt), ("paper", settled)):
    pnl = t["realized_pnl"]
    print(f"{name:9s}{len(t):4d} trades, {pnl.mean():+.3f} per trade, total {pnl.sum():+.1f}")
print(f"paper Brier score of the champion's probabilities: {brier:.3f}")

html = generate_report(paper, ws.artifacts_root / "report.html", title="pmlab tutorial", brier_score=brier)
print("report:", html.relative_to(root), f"({html.stat().st_size // 1024} KiB)")
settled.groupby("city_or_segment")["realized_pnl"].agg(trades="count", pnl="sum").round(2)

A few dozen trades are far too few to tell a good strategy from a lucky one, and that is the point of paper trading first: the champion records what it promised (`champion.json`, with the backtest's costs and reproducibility record), and `pmlab monitor` holds the live log against it.

## Where next

- Swap the sample for real markets: write or configure a plugin (`docs/plugin-authoring.md`) and run the same steps with `pmlab scan-markets`, `pmlab backtest`, `pmlab promote-champion`, `pmlab record-trades`, `pmlab settle-trades` and `pmlab report`.
- Calibrate the probabilities before trading them (`docs/calibration.md`), size by Kelly instead of a flat stake, and watch drift (`docs/drift-monitoring.md`).
- Re-run any backtest you published: `pmlab reproduce champion.json` (`docs/reproducibility.md`).